<a href="https://colab.research.google.com/github/SirLousy/LIS4693/blob/main/lab-2/Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Text Pre-Processing using spaCy

In this lab assignment, we will learn to perform some basic text pre-processing using spaCy.

*Note: Before starting this lab assignment, please complete the Introduction to spaCy notebook*

## Leaning Objectives

In this exercise, you will:

- Load and process your own text file (transcript.txt)
- Split text into sentences
- Count words and sentences
- Find frequently used words
- Use spaCy’s PhraseMatcher to find specific phrases
- Understand the difference between blank pipelines and full pipelines

This exercise builds directly on concepts from discussed in the precursor notebook on "Introduction to spaCy".

## Install spaCy

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

## Load and read your file from GitHub


In [ ]:
import requests

GITHUB_USERNAME = "SirLousy"
REPO_NAME = "LIS4693"
BRANCH = "main"
FILE_PATH = "transcript.txt"

raw_url = f"https://raw.githubusercontent.com/SirLousy/LIS4693/main/transcript.txt"
print("Raw URL:", raw_url)

r = requests.get(raw_url)
r.raise_for_status()
text = r.text

print("Loaded transcript. Preview (first 300 chars):")
print(text[:300])


Raw URL: https://raw.githubusercontent.com/SirLousy/LIS4693/main/transcript.txt
Loaded transcript. Preview (first 300 chars):
[Music]
[Music]
[Music]
back in smooth subside
[Music]
[Applause]
[Applause]
[Music]
sometimes I can be the true
my buddy
this may be love song
makes me feel some strange
[Music]
[Music]
[Applause]
[Music]
[Applause]
[Applause]
[Music]
[Music]
[Music]
[Music]
[Music]
[Music]
you
[Music]



In [ ]:
print("Number of characters:", len(text))
print("First 100 characters:\n", text[:100])


Number of characters: 288
First 100 characters:
 [Music]
[Music]
[Music]
back in smooth subside
[Music]
[Applause]
[Applause]
[Music]
sometimes I can


## Sentence Segmentation

Create a blank spaCy model and add sentencizer.

In [ ]:
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

doc = nlp(text)
sentences = list(doc.sents)

print("Number of sentences:", len(sentences))
print("\nFirst 3 sentences:\n")
for i, s in enumerate(sentences[:3], start=1):
    print(f"{i}. {s.text}\n")


Number of sentences: 1

First 3 sentences:

1. [Music]
[Music]
[Music]
back in smooth subside
[Music]
[Applause]
[Applause]
[Music]
sometimes I can be the true
my buddy
this may be love song
makes me feel some strange
[Music]
[Music]
[Applause]
[Music]
[Applause]
[Applause]
[Music]
[Music]
[Music]
[Music]
[Music]
[Music]
you
[Music]




## Word Count and Token Analysis

Let's count total words and unique words in your text file.

In [ ]:
words = [token.text.lower() for token in doc if token.is_alpha]
total_words = len(words)
unique_words = len(set(words))

print("Total number of words:", total_words)
print("Number of unique words:", unique_words)


Total number of words: 43
Number of unique words: 24


## Most Frequent Words

Let's find top 10 most frequent words in your file.

In [ ]:
from collections import Counter

word_freq = Counter(words)
most_common_word, most_common_count = word_freq.most_common(1)[0]

print("Most frequent word:", most_common_word)
print("Count:", most_common_count)

print("\nTop 10 most frequent words:")
for word, count in word_freq.most_common(10):
    print(word, count)


Most frequent word: music
Count: 15

Top 10 most frequent words:
music 15
applause 5
be 2
back 1
in 1
smooth 1
subside 1
sometimes 1
i 1
can 1


## Using Full spaCy Pipeline

Now use the full model for better linguistic analysis.

In [ ]:
try:
    nlp2 = spacy.load("en_core_web_sm")
except OSError:
    !python -m spacy download en_core_web_sm
    nlp2 = spacy.load("en_core_web_sm")

doc2 = nlp2(text)
entities = list(doc2.ents)
entity_types = sorted(set(ent.label_ for ent in entities))

print("How many named entities were found?:", len(entities))
print("What types of entities appear?:", entity_types)

print("\nFirst 15 entities (text - label):")
for ent in entities[:15]:
    print(ent.text, "-", ent.label_)


How many named entities were found?: 0
What types of entities appear?: []

First 15 entities (text - label):


## PhraseMatcher

In [ ]:
from spacy.matcher import PhraseMatcher
from collections import Counter

phrases = [
    "MUSIC",
    "Applause",
    "back in smooth subside"
]

matcher = PhraseMatcher(nlp2.vocab, attr="LOWER")
patterns = [nlp2.make_doc(p) for p in phrases]
matcher.add("TOPIC_PHRASES", patterns)

matches = matcher(doc2)
print("Total phrase matches found:", len(matches))

matched_spans = [doc2[start:end].text for _, start, end in matches]
match_counts = Counter([m.lower() for m in matched_spans])

print("\nMatches by phrase:")
for phrase, count in match_counts.most_common():
    print(f"{phrase} -> {count}")

print("\nExample match contexts (up to 10):")
for i, (_, start, end) in enumerate(matches[:10], start=1):
    span = doc2[start:end]
    left = doc2[max(0, start-8):start].text
    right = doc2[end:min(len(doc2), end+8)].text
    print(f"{i}. ...{left} [{span.text}] {right}...")


Total phrase matches found: 21

Matches by phrase:
music -> 15
applause -> 5
back in smooth subside -> 1

Example match contexts (up to 10):
1. ...[ [Music] ]
[Music]
[Music...
2. ...[Music]
[ [Music] ]
[Music]
back in...
3. ...Music]
[Music]
[ [Music] ]
back in smooth subside
[...
4. ...[Music]
[Music]
 [back in smooth subside] 
[Music]
[Applause]...
5. ...]
back in smooth subside
[ [Music] ]
[Applause]
[Applause...
6. ...smooth subside
[Music]
[ [Applause] ]
[Applause]
[Music...
7. ...Music]
[Applause]
[ [Applause] ]
[Music]
sometimes I...
8. ...Applause]
[Applause]
[ [Music] ]
sometimes I can be the true...
9. ...
makes me feel some strange
[ [Music] ]
[Music]
[Applause...
10. ...some strange
[Music]
[ [Music] ]
[Applause]
[Music...


## TASK 8: Reflection

Write a brief reflection (a few sentences):

- What went well?
- What did not go well / challenges?

**Reflection:**

(This lab went smoother once I fixed a couple mistakes from Day 1. I realized my files were not actually inside the lab-1 folder in my repo, which caused some confusion at first when trying to load the transcript. After organizing the repo correctly and using the proper raw GitHub link, everything started working the way it should. It felt good seeing the transcript load correctly and being able to break it into sentences and count the words.

One challenge I ran into was that the video I originally chose for Lab-1 was a music video. Because of that, there were not that many meaningful words being picked up during token analysis, which made some of the results less interesting. I also had to adjust my PhraseMatcher phrases to make sure they actually appeared in the transcript. Overall, this lab helped me understand how important file structure and input text quality are when doing text preprocessing..)


## Submission Checklist

1. Create a `lab-2` folder in your `lis4693` or `lis5693` GitHub repo.
2. In Colab: **File → Save a copy in GitHub** (do not upload manually).
3. Save your notebook into `lab-2/` (example: `lab-2/lab2.ipynb`).
4. Submit the link to your GitHub repo (or the `lab-2` folder) on Canvas.


# EXCERCISE

Open a new Google Colab notebook and complete the tasks below. As you work, add brief explanations using the **Text (Markdown) cells** throughout your notebook to describe what you are doing.

1. Make a new folder named `lab-2` in your `lis4693` or `lis5693` repo on GitHub. **[1 Point]**

2. Complete the following tasks:


**TASK 1**: Load and read your `transcript.txt` file from lab-1 from GitHub repo directly **[1 Point]**

**TASK 2**: How many characters are in your transcript? Print the first 100 characters. **[1 Point]**

**TASK 3**: Perform sentence segmentation using the blank pipeline
  - How many sentences are in your transcript? **[0.5 Point]**
  - Print the first 3 sentences **[0.5 Point]**

**TASK 4**: Perform Word Count and Token Analysis
  - What is the total number of words? **[0.5 Point]**
  - What is the number of unique words? **[0.5 Point]**

**TASK 5**: Find Most Frequent Words
  - What is the most frequent word? **[0.5 Point]**
  - Why do you think this word appears frequently? **[1 Point]**

**TASK 6**: Run full spaCy pipeline
  - How many named entities were found? **[0.5 Point]**
  - What types of entities appear? (PERSON, ORG, DATE, etc.) **[0.5 Point]**

**TASK 7**: Use PhraseMatcher **[1 Point]**

When you selected your YouTube video in Lab-1, what topic or subject were you interested in? Based on that topic, identify three specific phrases that are directly relevant to your search. For example, if your video was about information retrieval, relevant phrases might include "information retrieval," "text mining," and "data science."

Make sure the video you selected in Lab-1 was clearly related to your chosen topic and was the same video for which you downloaded the transcript in Lab-1.

**TASK 8**: At the end of your Colab notebook, create a new text cell and write a brief reflection for this assignment in a few sentences addressing the following **[2 Points]**:
 - What went well?
 - What did not go well or what challenges you encountered?

3. Push your Google Colab file to your `lab-2` GitHub repo from Colab. *No points will be given if you upload it to GitHub directly!* **[1 Point]**
4. Share the link to your `lab-2` GitHub repo for this lab assignment on CANVAS for credit. **[0.5 Point]**
